# Pre-Reqs:

In [1]:
import pandas as pd
import numpy as np
import optuna

from sklearn.preprocessing import LabelEncoder
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

# Data:


In [2]:
#Creating datframe from test and train csv's
TrainDataset = pd.read_csv('/content/train.csv')
TestDataset = pd.read_csv('/content/test.csv')

#Creating some additional dataframes to not be encoded
TrainNoEncDataset = pd.read_csv('/content/train.csv')
TestNoEncDataset = pd.read_csv('/content/test.csv')

In [3]:
TrainDataset.head()

,id,Age,Sex,Chest pain type,BP,Cholesterol,FBS over 120,EKG results,Max HR,Exercise angina,ST depression,Slope of ST,Number of vessels fluro,Thallium,Heart Disease
0,0,58,1,4,152,239,0,0,158,1,3.6,2,2,7,Presence
1,1,52,1,1,125,325,0,2,171,0,0.0,1,0,3,Absence
2,2,56,0,2,160,188,0,2,151,0,0.0,1,0,3,Absence
3,3,44,0,3,134,229,0,2,150,0,1.0,2,0,3,Absence
4,4,58,1,4,140,234,0,2,125,1,3.8,2,3,3,Presence


In [4]:
#Tested, no missing values

In [5]:
'''
Commented out to try a different form of encoding in CatBoost model

#Values needing to change to numerical in dataset
Features = ["Heart Disease"]

label_encoders = {}
for col in Features:
    le = LabelEncoder()

    # Make sure all values are strings
    TrainDataset[col] = TrainDataset[col].astype(str)

    # Fit on all unique SMILES
    all_smiles = pd.concat([TrainDataset[col]], axis=0).unique()
    le.fit(all_smiles)

    # Transform
    TrainDataset[col] = le.transform(TrainDataset[col])

    label_encoders[col] = le

#Numerical check
Train_Numerical = TrainDataset.applymap(lambda x: isinstance(x, (int, float))).all().all()
print(f"Test Dataset Numerical? {Train_Numerical}")
Test_Numerical = TestDataset.applymap(lambda x: isinstance(x, (int, float))).all().all()
print(f"Test Dataset Numerical? {Test_Numerical}")
'''

'\nCommented out to try a different form of encoding in CatBoost model\n\n#Values needing to change to numerical in dataset\nFeatures = ["Heart Disease"]\n\nlabel_encoders = {}\nfor col in Features:\n    le = LabelEncoder()\n\n    # Make sure all values are strings\n    TrainDataset[col] = TrainDataset[col].astype(str)\n\n    # Fit on all unique SMILES\n    all_smiles = pd.concat([TrainDataset[col]], axis=0).unique()\n    le.fit(all_smiles)\n\n    # Transform\n    TrainDataset[col] = le.transform(TrainDataset[col])\n\n    label_encoders[col] = le\n\n#Numerical check\nTrain_Numerical = TrainDataset.applymap(lambda x: isinstance(x, (int, float))).all().all()\nprint(f"Test Dataset Numerical? {Train_Numerical}")\nTest_Numerical = TestDataset.applymap(lambda x: isinstance(x, (int, float))).all().all()\nprint(f"Test Dataset Numerical? {Test_Numerical}")\n'

# Data Analysis:

# Model 1 CatBoost:

In [7]:
Target = "Heart Disease"
IDCol = "id"

X = TrainDataset.drop([Target, IDCol], axis=1)
y = TrainDataset[Target]
X_test = TestDataset.drop([IDCol], axis=1)

cat_features = X.select_dtypes(include=["object", "category"]).columns.tolist()

cv = StratifiedKFold(
    n_splits=7,
    shuffle=True,
    random_state=42
)

OOFPreds = np.zeros(len(X))
TestPreds = np.zeros(len(X_test))

nFolds = cv.get_n_splits()

for fold, (trn_idx, val_idx) in enumerate(cv.split(X, y)):
    print(f"\n--- Fold {fold + 1} ---")

    X_tr, X_val = X.iloc[trn_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[trn_idx], y.iloc[val_idx]

    # --- Detect potential flipped labels in the training fold ---
    baseline_model = CatBoostClassifier(
        iterations=500,
        learning_rate=0.05,
        depth=5,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=42,
        verbose=False
    )
    baseline_model.fit(X_tr, y_tr, cat_features=cat_features)
    y_tr_pred = baseline_model.predict_proba(X_tr)[:, 1]

    # Compute a weight vector: reduce weight for potential flipped labels
    # High confidence wrong predictions get lower weight (0.2)
    weight = np.ones(len(X_tr))
    threshold = 0.9  # adjust for strictness
    weight[(y_tr == 1) & (y_tr_pred < 1 - threshold)] = 0.2
    weight[(y_tr == 0) & (y_tr_pred > threshold)] = 0.2

    # --- Train the main model with weighted samples ---
    model = CatBoostClassifier(
        iterations=3000,
        learning_rate=0.03,
        depth=6,
        border_count=32,
        min_data_in_leaf=10,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=42,
        verbose=False
    )

    model.fit(
        X_tr, y_tr,
        sample_weight=weight,           # <-- use weights to downplay suspected flips
        eval_set=(X_val, y_val),
        cat_features=cat_features,
        early_stopping_rounds=300
    )

    OOFPreds[val_idx] = model.predict_proba(X_val)[:, 1]
    TestPreds += model.predict_proba(X_test)[:, 1] / nFolds

    fold_auc = roc_auc_score(y_val, OOFPreds[val_idx])
    print(f"Fold AUC: {fold_auc:.5f}")

overall_auc = roc_auc_score(y, OOFPreds)
print(f"\nOverall CV AUC: {overall_auc:.5f}")

submission = pd.DataFrame({
    "id": TestDataset[IDCol],
    "Heart Disease": TestPreds
})

submission.to_csv("KaggleSub1CatBoost_FlippedWeights.csv", index=False)



--- Fold 1 ---
Fold AUC: 0.95486

--- Fold 2 ---
Fold AUC: 0.95438

--- Fold 3 ---
Fold AUC: 0.95386

--- Fold 4 ---
Fold AUC: 0.95458

--- Fold 5 ---
Fold AUC: 0.95337

--- Fold 6 ---
Fold AUC: 0.95564

--- Fold 7 ---
Fold AUC: 0.95461

Overall CV AUC: 0.95447
